# Baseline Moses × A (SMT, 4M token)


In [ ]:
import json, re, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

DATASET = 'A'
CONFIG = 'natural'
BUDGET = 4_000_000

DATA = Path('/kaggle/input/tiki-matrix-data')
if not (DATA / 'quota_sampling.py').exists():
    hits = sorted(Path('/kaggle/input').rglob('quota_sampling.py'))
    if not hits:
        raise SystemExit('dataset not mounted: ' + str(list(Path('/kaggle/input').iterdir())))
    DATA = hits[0].parent
sys.path.insert(0, str(DATA))
from quota_sampling import quota_indices
print('DATA', DATA)

units = pd.read_parquet(DATA / f'units_{DATASET}.parquet')
meta = json.loads((DATA / f'meta_{DATASET}.json').read_text(encoding='utf-8'))
cfg = meta['configs'].get(CONFIG)
idx, actual = quota_indices(units, cfg, 42, BUDGET)
sample = units.iloc[idx].reset_index(drop=True)
eval_frame = pd.read_parquet(DATA / f'eval_{DATASET}.parquet')
print('sample', len(sample), 'tokens', round(sample.eff_tokens.sum()/1e6, 3), actual)
print('eval', len(eval_frame))

CJK = re.compile(r'[\u4e00-\u9fff]')
PUNCT = re.compile(r'([,.;:!?()\[\]{}"“”‘’、，。；：！？（）【】《》])')

def tok_vi(text: str) -> str:
    t = str(text or '').strip()
    t = PUNCT.sub(r' \1 ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tok_zh(text: str) -> str:
    t = str(text or '').strip()
    t = PUNCT.sub(r' \1 ', t)
    t = re.sub(r'([\u4e00-\u9fff])', r' \1 ', t)
    return re.sub(r'\s+', ' ', t).strip()

WORK = Path('/kaggle/working/work')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'train.vi').write_text('\n'.join(tok_vi(x) for x in sample.source_text) + '\n', encoding='utf-8')
(WORK / 'train.zh').write_text('\n'.join(tok_zh(x) for x in sample.target_text) + '\n', encoding='utf-8')
(WORK / 'eval.vi').write_text('\n'.join(tok_vi(x) for x in eval_frame.source) + '\n', encoding='utf-8')
print('wrote train/eval text')


In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential git libboost-all-dev zlib1g-dev libbz2-dev cmake liblzma-dev > /kaggle/working/apt.log 2>&1
cd /kaggle/working
[ -d mosesdecoder ] || git clone -q --depth 1 https://github.com/moses-smt/mosesdecoder.git
cd mosesdecoder
./bjam -j4 --with-boost=/usr > /kaggle/working/moses_build.log 2>&1
ls -l bin/moses bin/lmplz


In [ ]:
%%bash
set -e
pip install -q eflomal
cd /kaggle/working/work
python - <<'PY'
import time
from eflomal import Aligner
t = time.time()
al = Aligner(model=3, n_iterations=(4, 4, 4), n_samplers=4)
with open('train.vi', encoding='utf-8') as src, open('train.zh', encoding='utf-8') as trg:
    al.align(src, trg, links_filename_fwd='fwd', links_filename_rev='rev', quiet=True)
print('align sec', round(time.time() - t, 1))
PY
python - <<'PY'
from pathlib import Path
fwd = Path('fwd').read_text().splitlines()
rev = Path('rev').read_text().splitlines()
lines = []
for a, b in zip(fwd, rev):
    links = set(a.split()) | set(b.split())
    lines.append(' '.join(sorted(links, key=lambda x: (int(x.split('-')[0]), int(x.split('-')[1])))))
Path('sym.grow-diag-final-and').write_text('\n'.join(lines), encoding='utf-8')
print('sym lines', len(lines), 'sample', (lines[0][:80] if lines else ''))
PY


In [ ]:
%%bash
set -e
cd /kaggle/working/work
[ -f lm.arpa.gz ] || { zcat -f train.zh | /kaggle/working/mosesdecoder/bin/lmplz -o 3 > lm.arpa 2> lm.log; gzip -f lm.arpa; }
ls -lh lm.arpa.gz
mkdir -p work/model work/corpus work/phrase-model work/lexical-reordering
if ! /kaggle/working/mosesdecoder/scripts/training/train-model.perl \
  -root-dir /kaggle/working/work/work \
  -corpus /kaggle/working/work/train \
  -f vi -e zh \
  -alignment grow-diag-final-and -alignment-file /kaggle/working/work/sym \
  -do-steps 4-9 \
  -lm 0:3:/kaggle/working/work/lm.arpa.gz \
  -max-phrase-length 7 \
  -reordering msd-bidirectional-fe \
  > /kaggle/working/work/train_model.log 2>&1; then
  echo "TRAIN MODEL FAILED"; tail -40 /kaggle/working/work/train_model.log; exit 1
fi
tail -8 /kaggle/working/work/train_model.log
ls -l /kaggle/working/work/work/model/moses.ini


In [ ]:
%%bash
set -e
cd /kaggle/working/work
/kaggle/working/mosesdecoder/bin/moses -f work/model/moses.ini -threads 4 \
  < eval.vi > pred.zh 2> moses_decode.log
wc -l pred.zh
tail -5 moses_decode.log
[ "$(wc -l < pred.zh)" -ge 1500 ] || { echo "DECODE SHORT"; exit 1; }


In [ ]:
import pandas as pd

def detok(text: str) -> str:
    t = re.sub(r'\s+', ' ', str(text or '')).strip()
    t = re.sub(r'(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])', '', t)
    t = re.sub(r'\s+(?=[，。！？；：、）】》])', '', t)
    t = re.sub(r'(?<=[（【《])\s+', '', t)
    return t

pred_path = Path('/kaggle/working/work/pred.zh')
hyps = [l.rstrip('\n') for l in pred_path.read_text(encoding='utf-8').splitlines()]
if len(hyps) < len(eval_frame):
    hyps += [''] * (len(eval_frame) - len(hyps))
out = eval_frame[['eval_id', 'dataset', 'pair_type', 'source', 'reference']].copy()
out['hypothesis'] = [detok(h) for h in hyps[:len(out)]]
out['config'] = 'moses_' + DATASET.lower()
out['seed'] = 42
out.to_parquet(f'/kaggle/working/hyp_{out.config.iloc[0]}_seed42.parquet', index=False)
print(out.head(3).to_string()[:500])
print('empty', float((out.hypothesis.str.len() == 0).mean()), 'n', len(out))
